# 03_feature_engineering

In [1]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nChecking required files:")

files_to_check = [
    "../data/processed/cleaned_dataset.csv",
    "../data/raw/dataset.csv"
]

for file in files_to_check:
    print(file, ":", os.path.exists(file))

Current working directory:
C:\Users\AFRIN\474 Classroom Practice\ANN_PROJECT\notebooks

Checking required files:
../data/processed/cleaned_dataset.csv : True
../data/raw/dataset.csv : True


## Importing libraries

In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import MultiLabelBinarizer

print("Libraries imported successfully!")

Libraries imported successfully!


## Loading the cleaned dataset

In [11]:
df = pd.read_csv("../data/processed/cleaned_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (4920, 18)


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Define target column
target_column = "Disease"

# Identify all symptom columns
symptom_columns = [
    col for col in df.columns
    if col.startswith("Symptom_")
]

print("Target column:", target_column)
print("Number of symptom columns:", len(symptom_columns))
print("\nSymptom columns:")
print(symptom_columns)

Target column: Disease
Number of symptom columns: 17

Symptom columns:
['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']


In [9]:
# STEP 1: Identify symptom columns
target_column = "Disease"

symptom_columns = [
    col for col in df.columns
    if col.startswith("Symptom_")
]

# STEP 2: Replace missing symptom values
df[symptom_columns] = df[symptom_columns].fillna("")

# STEP 3: Verify missing values
missing_values = df[symptom_columns].isnull().sum().sum()

print("Missing values after cleaning:", missing_values)

# STEP 4: Check dataset
print("\nDataset shape:", df.shape)

# STEP 5: Count symptoms in each record
df["Total_Symptoms"] = (
    df[symptom_columns].ne("").sum(axis=1)
)

print("\nSymptom count summary:")
print(df["Total_Symptoms"].describe())

print("\nRecords by symptom count:")
print(df["Total_Symptoms"].value_counts().sort_index())

Missing values after cleaning: 0

Dataset shape: (4920, 18)

Symptom count summary:
count    4920.000000
mean        7.448780
std         3.592166
min         3.000000
25%         5.000000
50%         6.000000
75%        10.000000
max        17.000000
Name: Total_Symptoms, dtype: float64

Records by symptom count:
Total_Symptoms
3     348
4     858
5     780
6     666
7     324
8     252
9     180
10    318
11    450
12    240
13    198
14     66
15     48
16    120
17     72
Name: count, dtype: int64


In [10]:
unique_df = df.drop_duplicates().reset_index(drop=True)

print("Original records:", len(df))
print("Unique records:", len(unique_df))
print("Duplicates removed:", len(df) - len(unique_df))

print("\nDisease classes:", unique_df["Disease"].nunique())

Original records: 4920
Unique records: 304
Duplicates removed: 4616

Disease classes: 41


### Creating a list of symptoms for every record

In [12]:
symptom_lists = []

for _, row in unique_df[symptom_columns].iterrows():

    symptoms = [
        symptom
        for symptom in row
        if symptom != ""
    ]

    symptom_lists.append(symptoms)

unique_df["Symptoms_List"] = symptom_lists

unique_df[["Disease", "Symptoms_List"]].head(10)

,Disease,Symptoms_List
0,fungal infection,"[itching, skin_rash, nodal_skin_eruptions, dis..."
1,fungal infection,"[skin_rash, nodal_skin_eruptions, dischromic _..."
2,fungal infection,"[itching, nodal_skin_eruptions, dischromic _pa..."
3,fungal infection,"[itching, skin_rash, dischromic _patches]"
4,fungal infection,"[itching, skin_rash, nodal_skin_eruptions]"
5,allergy,"[continuous_sneezing, shivering, chills, water..."
6,allergy,"[shivering, chills, watering_from_eyes]"
7,allergy,"[continuous_sneezing, chills, watering_from_eyes]"
8,allergy,"[continuous_sneezing, shivering, watering_from..."
9,allergy,"[continuous_sneezing, shivering, chills]"


### Applying Multi-Hot Encoding

In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

X = mlb.fit_transform(unique_df["Symptoms_List"])

X = pd.DataFrame(
    X,
    columns=mlb.classes_,
    index=unique_df.index
)

print("Multi-Hot Encoding completed!")

print("Feature matrix shape:", X.shape)

print("\nFirst 5 encoded records:")
X.head()

Multi-Hot Encoding completed!
Feature matrix shape: (304, 131)

First 5 encoded records:


,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,vomiting,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Understanding the encoded features

In [14]:
print("Available symptom features:")
print(list(X.columns))

print("\nNumber of features:", X.shape[1])

print("\nUnique values in encoded data:")
print(np.unique(X.values))

Available symptom features:
['abdominal_pain', 'abnormal_menstruation', 'acidity', 'acute_liver_failure', 'altered_sensorium', 'anxiety', 'back_pain', 'belly_pain', 'blackheads', 'bladder_discomfort', 'blister', 'blood_in_sputum', 'bloody_stool', 'blurred_and_distorted_vision', 'breathlessness', 'brittle_nails', 'bruising', 'burning_micturition', 'chest_pain', 'chills', 'cold_hands_and_feets', 'coma', 'congestion', 'constipation', 'continuous_feel_of_urine', 'continuous_sneezing', 'cough', 'cramps', 'dark_urine', 'dehydration', 'depression', 'diarrhoea', 'dischromic _patches', 'distention_of_abdomen', 'dizziness', 'drying_and_tingling_lips', 'enlarged_thyroid', 'excessive_hunger', 'extra_marital_contacts', 'family_history', 'fast_heart_rate', 'fatigue', 'fluid_overload', 'foul_smell_of urine', 'headache', 'high_fever', 'hip_joint_pain', 'history_of_alcohol_consumption', 'increased_appetite', 'indigestion', 'inflammatory_nails', 'internal_itching', 'irregular_sugar_level', 'irritability

### Preparing the target variable

In [15]:
y = unique_df["Disease"].reset_index(drop=True)

print("Target variable created!")

print("Target shape:", y.shape)

print("\nNumber of disease classes:", y.nunique())

print("\nDisease distribution:")
print(y.value_counts())

Target variable created!
Target shape: (304,)

Number of disease classes: 41

Disease distribution:
Disease
migraine                                   10
dengue                                     10
hepatitis d                                10
chicken pox                                10
typhoid                                     9
hepatitis b                                 9
jaundice                                    9
hepatitis a                                 9
diabetes                                    9
hepatitis e                                 9
pneumonia                                   9
common cold                                 9
hypoglycemia                                9
hyperthyroidism                             9
tuberculosis                                9
alcoholic hepatitis                         8
chronic cholestasis                         8
varicose veins                              8
hypothyroidism                              8
malaria           

### Checking the number of unique records per disease

In [16]:
disease_counts = y.value_counts()

print("Minimum records per disease:",
      disease_counts.min())

print("\nMaximum records per disease:",
      disease_counts.max())

print("\nDiseases with fewer than 5 records:")
print(disease_counts[disease_counts < 5])

Minimum records per disease: 5

Maximum records per disease: 10

Diseases with fewer than 5 records:
Series([], Name: count, dtype: int64)


### Combining features and target

In [17]:
feature_df = X.copy()

feature_df["Disease"] = y.values

print("Final feature dataset shape:", feature_df.shape)

feature_df.head()

Final feature dataset shape: (304, 132)


,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin,Disease
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,fungal infection
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,fungal infection
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,fungal infection
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,fungal infection
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,fungal infection


### Save the files

In [18]:
import os
import joblib

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save encoded dataset
feature_df.to_csv(
    "../data/processed/feature_dataset.csv",
    index=False
)

# Save unique cleaned dataset
unique_df.drop(
    columns=["Symptoms_List"],
    errors="ignore"
).to_csv(
    "../data/processed/unique_cleaned_dataset.csv",
    index=False
)

# Save symptom encoder
joblib.dump(
    mlb,
    "../models/symptom_binarizer.pkl"
)

print("Files saved successfully!")

Files saved successfully!


### Final summary

In [19]:
print("=" * 45)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 45)

print("Original records:", len(df))
print("Unique records:", len(unique_df))
print("Duplicate records removed:", len(df) - len(unique_df))

print("Disease classes:", y.nunique())
print("Symptom features:", X.shape[1])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Final dataset shape:", feature_df.shape)

print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

print("\nFeature engineering completed!")

FEATURE ENGINEERING SUMMARY
Original records: 4920
Unique records: 304
Duplicate records removed: 4616
Disease classes: 41
Symptom features: 131
Feature matrix shape: (304, 131)
Target shape: (304,)
Final dataset shape: (304, 132)
Missing values in X: 0
Missing values in y: 0

Feature engineering completed!
